# File Handling: Solutions

Instructor copy. Run the setup cell first, then any solution below.

In [ ]:
# SETUP: recreate the sample files this notebook needs
import os, json, random, csv, glob, shutil
from datetime import date

os.makedirs("datasets", exist_ok=True)

with open("example.txt", "w", encoding="utf-8") as f:
    f.write("Lorem ipsum dolor sit amet\n")
    f.write("ahmed is a chelsea fan\n")
    f.write("This will append to the file.")

with open("sales.csv", "w", encoding="utf-8") as f:
    f.write("order_id,customer,item,amount,country\n")
    f.write("1,ahmed,laptop,450000,NG\n")
    f.write("2,lahla,phone,180000,GH\n")
    f.write("3,ahmed,mouse,8000,NG\n")
    f.write("4,judith,laptop,520000,NG\n")
    f.write("5,uche,monitor,95000,KE\n")

with open("messy_sales.csv", "w", encoding="utf-8") as f:
    f.write("order_id,customer,item,amount,country\n")
    f.write("6,ahmed,keyboard,15000,NG\n")
    f.write("7,,phone,180000,GH\n")
    f.write("8,judith,laptop,N/A,NG\n")
    f.write("9,uche,monitor,,KE\n")
    f.write("10,lahla,mouse,7500,\n")
    f.write("6,ahmed,keyboard,15000,NG\n")

with open("config.json", "w", encoding="utf-8") as f:
    json.dump({
        "source": "postgres",
        "tables": ["customers", "orders", "payments"],
        "batch_size": 5000,
        "checks": {"null_check": True, "row_count_check": True},
    }, f, indent=2)

random.seed(42)
with open("datasets/another_sales_records.csv", "w", encoding="utf-8") as f:
    f.write("Region,Country,Units Sold,Total Revenue\n")
    for i in range(60):
        f.write(f"Region{i % 4},Country{i % 7},{random.randint(500, 9000)},{random.randint(1000, 99999)}\n")

for month in ["08", "09"]:
    path = os.path.join("landing", "2026", month)
    os.makedirs(path, exist_ok=True)
    with open(os.path.join(path, "orders.csv"), "w", encoding="utf-8") as f:
        f.write("order_id,amount\n1,100\n2,250\n")

print("setup done")

## Reading and Writing: tasks

In [ ]:
# Task 1: write then read
with open("my_notes.txt", "w", encoding="utf-8") as f:
    f.write("with closes the file for me\n")
    f.write("w wipes the file, a appends\n")
    f.write("always pass encoding utf-8\n")

with open("my_notes.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Task 2: append
with open("my_notes.txt", "a", encoding="utf-8") as f:
    f.write("csv values are always strings\n")
    f.write("write to a temp file, then rename\n")

with open("my_notes.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Task 3: counts
with open("example.txt", "r", encoding="utf-8") as f:
    contents = f.read()

print(f"characters: {len(contents)}")
print(f"lines:      {len(contents.splitlines())}")

In [ ]:
# Task 4: line by line, uppercase, numbered
with open("example.txt", "r", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(f"{number}: {line.strip().upper()}")

In [ ]:
# Task 5: list to file and back
students = ["ahmed", "lahla", "judith", "uche"]

with open("students.txt", "w", encoding="utf-8") as f:
    for name in students:
        f.write(name + "\n")

with open("students.txt", "r", encoding="utf-8") as f:
    loaded = [line.strip() for line in f]

print(loaded)
print(f"same as the original? {loaded == students}")

## CSV and JSON: tasks

In [ ]:
# Task 1: total the amount column
import csv

total = 0
with open("sales.csv", "r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        total += int(row["amount"])

print(f"total: {total:,}")

In [ ]:
# Task 2: DictWriter
import csv

students = [
    {"name": "ahmed", "score": 78, "track": "data engineering"},
    {"name": "lahla", "score": 91, "track": "analytics"},
    {"name": "judith", "score": 64, "track": "data engineering"},
]

with open("students.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score", "track"])
    writer.writeheader()
    writer.writerows(students)

with open("students.csv", "r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        print(row)

In [ ]:
# Task 3: read the config
import json

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print(f"{len(config['tables'])} table(s):")
for table in config["tables"]:
    print(f"  {table}")

In [ ]:
# DE Task 4: validate the messy file
import csv

clean = []
rejected = []
seen_ids = set()

with open("messy_sales.csv", "r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        if row["order_id"] in seen_ids:
            rejected.append({**row, "reason": "duplicate order_id"})
            continue
        if not row["customer"]:
            rejected.append({**row, "reason": "empty customer"})
            continue
        if not row["country"]:
            rejected.append({**row, "reason": "empty country"})
            continue
        try:
            float(row["amount"])
        except ValueError:
            rejected.append({**row, "reason": f"bad amount {row['amount']!r}"})
            continue
        seen_ids.add(row["order_id"])
        clean.append(row)

print(f"clean: {len(clean)}, rejected: {len(rejected)}")
for bad in rejected:
    print(f"  order {bad['order_id']}: {bad['reason']}")

In [ ]:
# DE Task 5: write both outputs
import csv

with open("clean_sales.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "customer", "item", "amount", "country"])
    writer.writeheader()
    writer.writerows(clean)

with open("rejected_sales.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f, fieldnames=["order_id", "customer", "item", "amount", "country", "reason"]
    )
    writer.writeheader()
    writer.writerows(rejected)

print("clean_sales.csv:")
with open("clean_sales.csv", encoding="utf-8") as f:
    print(f.read())

print("rejected_sales.csv:")
with open("rejected_sales.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# DE Task 6: csv to jsonl with real numbers
import csv, json

with open("sales.csv", "r", encoding="utf-8", newline="") as source:
    with open("sales.jsonl", "w", encoding="utf-8") as target:
        for row in csv.DictReader(source):
            row["order_id"] = int(row["order_id"])
            row["amount"] = float(row["amount"])
            target.write(json.dumps(row) + "\n")

with open("sales.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        print(f"{record['order_id']} {record['customer']} {record['amount']} "
              f"({type(record['amount']).__name__})")

In [ ]:
# DE Task 7: count rows without loading the file
path = "datasets/another_sales_records.csv"

with open(path, "r", encoding="utf-8") as f:
    header = f.readline().strip()
    rows = 0
    for line in f:
        rows += 1

print(f"header: {header}")
print(f"data rows: {rows}")

## File errors: tasks

In [ ]:
# Task 1: safe_read
def safe_read(path):
    """Return the file contents, or None with a clear message."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"{path}: not found")
    except PermissionError:
        print(f"{path}: no permission to read")
    return None

print(len(safe_read("example.txt") or ""))
print(safe_read("nope.txt"))

In [ ]:
# Task 2: x mode refuses to overwrite
try:
    with open("my_notes.txt", "x", encoding="utf-8") as f:
        f.write("never written")
except FileExistsError:
    print("my_notes.txt already exists and is protected from being overwritten")

In [ ]:
# DE Task 3: row_count without loading the file
def row_count(path, has_header=True):
    """Count the data rows in a text file, one line at a time."""
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        if has_header:
            f.readline()
        for line in f:
            if line.strip():
                count += 1
    return count

print(row_count("sales.csv"))
print(row_count("datasets/another_sales_records.csv"))

In [ ]:
# DE Task 4: merge every csv matching a pattern
import csv, glob, os

def merge_csvs(pattern, output_path):
    """Merge every CSV matching the pattern into one file, stamped with its source."""
    all_rows = []
    fieldnames = None

    for path in sorted(glob.glob(pattern, recursive=True)):
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            if fieldnames is None:
                fieldnames = list(reader.fieldnames) + ["source_file"]
            for row in reader:
                row["source_file"] = path
                all_rows.append(row)

    if not all_rows:
        raise ValueError(f"no files matched {pattern!r}")

    with open(output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    return len(all_rows)

count = merge_csvs("landing/**/*.csv", "merged.csv")
print(f"merged {count} row(s)")

with open("merged.csv", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# DE Task 5: the atomic version
import csv, os

def write_csv_atomically(path, rows, fieldnames):
    """Write to a temp file, then rename, so readers never see a partial file."""
    temp_path = path + ".tmp"
    with open(temp_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp_path, path)
    return path

write_csv_atomically(
    "clean_sales.csv", clean, ["order_id", "customer", "item", "amount", "country"]
)
print(f"written atomically. leftover temp file? {os.path.exists('clean_sales.csv.tmp')}")

## Mini project: full worked solution

In [ ]:
import csv, json, os
from datetime import date

FIELDS = ["order_id", "customer", "item", "amount", "country"]


def read_rows(path):
    """Read a CSV into a list of dicts."""
    with open(path, "r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def validate_row(row, seen_ids):
    """Return (True, "") for a good row, or (False, reason)."""
    if row["order_id"] in seen_ids:
        return False, "duplicate order_id"
    if not row["customer"].strip():
        return False, "empty customer"
    if not row["country"].strip():
        return False, "empty country"
    try:
        float(row["amount"])
    except ValueError:
        return False, f"amount {row['amount']!r} is not a number"
    return True, ""


def transform_row(row):
    """Convert types and add the load timestamp."""
    return {
        "order_id": int(row["order_id"]),
        "customer": row["customer"].strip().upper(),
        "item": row["item"].strip(),
        "amount": float(row["amount"]),
        "country": row["country"].strip().upper(),
        "loaded_at": date.today().isoformat(),
    }


def write_atomically(path, rows, fieldnames):
    """Write a CSV via a temp file, then rename."""
    temp_path = path + ".tmp"
    with open(temp_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp_path, path)


print("functions defined")

In [ ]:
def run(input_path, write=True):
    """Run the whole pipeline. Set write=False for a dry run."""
    raw = read_rows(input_path)

    clean = []
    rejected = []
    seen_ids = set()

    for row in raw:
        ok, reason = validate_row(row, seen_ids)
        if not ok:
            rejected.append({**row, "reason": reason})
            continue
        seen_ids.add(row["order_id"])
        clean.append(transform_row(row))

    total = sum(row["amount"] for row in clean)

    summary = {
        "run_date": date.today().isoformat(),
        "source": input_path,
        "rows_in": len(raw),
        "rows_loaded": len(clean),
        "rows_rejected": len(rejected),
        "rejection_rate_pct": round(len(rejected) / len(raw) * 100, 1) if raw else 0,
        "total_amount": total,
        "reasons": [f"order {r['order_id']}: {r['reason']}" for r in rejected],
        "dry_run": not write,
    }

    if write:
        write_atomically("clean_sales.csv", clean, FIELDS + ["loaded_at"])
        write_atomically("rejected_sales.csv", rejected, FIELDS + ["reason"])
        with open("run_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)

        with open("clean_sales.jsonl", "w", encoding="utf-8") as f:
            for row in clean:
                f.write(json.dumps(row) + "\n")

    return clean, rejected, summary


def report(summary):
    """Print the run summary in a form a human will read."""
    print("=== INGESTION REPORT ===")
    if summary["dry_run"]:
        print("(DRY RUN, nothing was written)")
    print(f"Source:         {summary['source']}")
    print(f"Run date:       {summary['run_date']}")
    print(f"Rows in:        {summary['rows_in']}")
    print(f"Rows loaded:    {summary['rows_loaded']}")
    print(f"Rows rejected:  {summary['rows_rejected']}")
    for reason in summary["reasons"]:
        print(f"   -> {reason}")
    print(f"Rejection rate: {summary['rejection_rate_pct']}%")
    print(f"Total amount:   {summary['total_amount']:,.2f}")


clean, rejected, summary = run("messy_sales.csv")
report(summary)

In [ ]:
# Stretch 6: verify our own output
import csv

with open("clean_sales.csv", "r", encoding="utf-8", newline="") as f:
    written = list(csv.DictReader(f))

written_total = sum(float(row["amount"]) for row in written)

print(f"summary said {summary['rows_loaded']} rows, file has {len(written)}")
print(f"summary said {summary['total_amount']:,.2f}, file totals {written_total:,.2f}")

if len(written) == summary["rows_loaded"] and abs(written_total - summary["total_amount"]) < 0.01:
    print("VERIFIED: output matches the summary")
else:
    print("MISMATCH: investigate before anyone uses this data")

In [ ]:
# Stretch 7 and 8: jsonl output and the dry run
with open("clean_sales.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

print()
_, _, dry_summary = run("messy_sales.csv", write=False)
report(dry_summary)

In [ ]:
# Clean up
import os, shutil

for name in ["example.txt", "sales.csv", "messy_sales.csv", "config.json",
             "my_notes.txt", "students.txt", "students.csv", "sales.jsonl",
             "clean_sales.csv", "rejected_sales.csv", "clean_sales.jsonl",
             "run_summary.json", "merged.csv"]:
    if os.path.isfile(name):
        os.remove(name)

for folder in ["landing", "datasets"]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)

print("cleaned up")